In [9]:
import pandas as pd
import os

In [10]:
df_compilado = pd.read_csv(r"C:\Arquivos Vsco\Projeto\Challenge Orale\data\csv a ser tratado\compilado_challenge_completo 1.csv", sep=";")

# Converte "10840,0" (texto, vírgula decimal) -> 10840.0 (float)
df_compilado["Valor"] = (
    df_compilado["Valor"]
    .str.replace(".", "", regex=False)   # remove separador de milhar, se houver
    .str.replace(",", ".", regex=False)  # vírgula decimal -> ponto
    .astype(float)
)

# Extrai código IBGE da UF, separado do nome (mesma lógica que já usamos)
df_compilado["CO_UF_IBGE"] = df_compilado["Unidade_Federacao"].str.extract(r"^(\d+)")
df_compilado["NOME_UF"] = df_compilado["Unidade_Federacao"].str.replace(r"^\d+\s*", "", regex=True)

print(df_compilado.dtypes)
print(df_compilado.head())

Ano                    int64
Mes                  float64
Fonte                 object
Indicador             object
Unidade_Federacao     object
Valor                float64
Arquivo_Origem        object
CO_UF_IBGE            object
NOME_UF               object
dtype: object
    Ano  Mes Fonte    Indicador Unidade_Federacao    Valor  \
0  2023  4.0   SIH  Internações       11 Rondônia  10840.0   
1  2023  4.0   SIH  Internações           12 Acre   5009.0   
2  2023  4.0   SIH  Internações       13 Amazonas  17087.0   
3  2023  4.0   SIH  Internações        14 Roraima   4492.0   
4  2023  4.0   SIH  Internações           15 Pará  43939.0   

                         Arquivo_Origem CO_UF_IBGE   NOME_UF  
0  sih_cnv_niuf085834187_72_164_185.csv         11  Rondônia  
1  sih_cnv_niuf085834187_72_164_185.csv         12      Acre  
2  sih_cnv_niuf085834187_72_164_185.csv         13  Amazonas  
3  sih_cnv_niuf085834187_72_164_185.csv         14   Roraima  
4  sih_cnv_niuf085834187_72_164_185.c

In [11]:
# Separa o que é ANUAL (CNES, IBGE - não tem mês) do que é MENSAL (SIH, Obitos - tem mês)
df_anual = df_compilado[df_compilado["Fonte"].isin(["CNES", "IBGE"])]
df_mensal = df_compilado[df_compilado["Fonte"].isin(["SIH", "Obitos"])]

# Pivota o anual: uma linha por UF/Ano, colunas = cada indicador
pivot_anual = df_anual.pivot_table(
    index=["NOME_UF", "CO_UF_IBGE", "Ano"],
    columns="Indicador",
    values="Valor"
).reset_index()

# Pivota o mensal: uma linha por UF/Ano/Mes, colunas = cada indicador
pivot_mensal = df_mensal.pivot_table(
    index=["NOME_UF", "CO_UF_IBGE", "Ano", "Mes"],
    columns="Indicador",
    values="Valor"
).reset_index()

print(pivot_anual.head())
print(pivot_anual.shape)
print()
print(pivot_mensal.head())
print(pivot_mensal.shape)

Indicador  NOME_UF CO_UF_IBGE   Ano  Leitos SUS  Leitos UTI total  \
0             Acre         12  2023      1547.0             116.0   
1             Acre         12  2024      1611.0             106.0   
2             Acre         12  2025      1694.0             105.0   
3          Alagoas         27  2023      5997.0             694.0   
4          Alagoas         27  2024      6062.0             697.0   

Indicador  Leitos existentes  População residente  Qtd hospitais  
0                     1790.0             876582.0           34.0  
1                     1793.0             880631.0           34.0  
2                     1881.0             884372.0           34.0  
3                     7338.0            3218607.0           98.0  
4                     7413.0            3220104.0           97.0  
(81, 8)

Indicador NOME_UF CO_UF_IBGE   Ano  Mes  Dias permanência  Internações  \
0            Acre         12  2023  1.0           17664.0       4164.0   
1            Acre         

In [12]:
# Agrega o SIH mensal para anual (Internações, Óbitos, etc. -> soma faz sentido, são contagens/valores por período)
# Média permanência NÃO deve ser somada -- é uma média, então recalculamos como média do ano
sih_anual = pivot_mensal.groupby(["NOME_UF", "CO_UF_IBGE", "Ano"]).agg(
    TOTAL_INTERNACOES=("Internações", "sum"),
    TOTAL_OBITOS_SIH=("Óbitos", "sum"),
    VALOR_TOTAL_SIH=("Valor total", "sum"),
    MEDIA_PERMANENCIA_ANO=("Média permanência", "mean")  # média das médias mensais
).reset_index()

# Junta anual (CNES+IBGE) com o anual do SIH
df_final_uf = pivot_anual.merge(sih_anual, on=["NOME_UF", "CO_UF_IBGE", "Ano"], how="left")

print(df_final_uf.shape)
print(df_final_uf.head())
print()
print("Nulos por coluna:")
print(df_final_uf.isnull().sum())

(81, 12)
   NOME_UF CO_UF_IBGE   Ano  Leitos SUS  Leitos UTI total  Leitos existentes  \
0     Acre         12  2023      1547.0             116.0             1790.0   
1     Acre         12  2024      1611.0             106.0             1793.0   
2     Acre         12  2025      1694.0             105.0             1881.0   
3  Alagoas         27  2023      5997.0             694.0             7338.0   
4  Alagoas         27  2024      6062.0             697.0             7413.0   

   População residente  Qtd hospitais  TOTAL_INTERNACOES  TOTAL_OBITOS_SIH  \
0             876582.0           34.0            59780.0            1786.0   
1             880631.0           34.0            57240.0            1805.0   
2             884372.0           34.0            61479.0            1791.0   
3            3218607.0           98.0           162320.0            7926.0   
4            3220104.0           97.0           170203.0            7776.0   

   VALOR_TOTAL_SIH  MEDIA_PERMANENCIA_ANO

In [13]:
df_final_uf["INTERNACOES_POR_LEITO"] = df_final_uf["TOTAL_INTERNACOES"] / df_final_uf["Leitos existentes"]
df_final_uf["TAXA_MORTALIDADE_HOSPITALAR"] = (df_final_uf["TOTAL_OBITOS_SIH"] / df_final_uf["TOTAL_INTERNACOES"]) * 100
df_final_uf["GASTO_MEDIO_INTERNACAO"] = df_final_uf["VALOR_TOTAL_SIH"] / df_final_uf["TOTAL_INTERNACOES"]
df_final_uf["LEITOS_POR_10K_HAB"] = (df_final_uf["Leitos existentes"] / df_final_uf["População residente"]) * 10000

# Correlação entre as métricas -- item obrigatório do enunciado
print(df_final_uf[["LEITOS_POR_10K_HAB", "INTERNACOES_POR_LEITO", "TAXA_MORTALIDADE_HOSPITALAR", "GASTO_MEDIO_INTERNACAO"]].corr())

                             LEITOS_POR_10K_HAB  INTERNACOES_POR_LEITO  \
LEITOS_POR_10K_HAB                     1.000000              -0.555631   
INTERNACOES_POR_LEITO                 -0.555631               1.000000   
TAXA_MORTALIDADE_HOSPITALAR            0.056663              -0.343205   
GASTO_MEDIO_INTERNACAO                 0.207771              -0.044229   

                             TAXA_MORTALIDADE_HOSPITALAR  \
LEITOS_POR_10K_HAB                              0.056663   
INTERNACOES_POR_LEITO                          -0.343205   
TAXA_MORTALIDADE_HOSPITALAR                     1.000000   
GASTO_MEDIO_INTERNACAO                          0.673361   

                             GASTO_MEDIO_INTERNACAO  
LEITOS_POR_10K_HAB                         0.207771  
INTERNACOES_POR_LEITO                     -0.044229  
TAXA_MORTALIDADE_HOSPITALAR                0.673361  
GASTO_MEDIO_INTERNACAO                     1.000000  


In [14]:
os.makedirs("data/processed", exist_ok=True)

df_final_uf.to_csv("data/processed/dataset_unificado_uf_2023_2025.csv", index=False, encoding="utf-8")
print("Salvo:", df_final_uf.shape)

Salvo: (81, 16)
